### Librerías a utilizar
---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

### Importación de datos
---

In [2]:
df = pd.read_csv('../data/raw/synthetic_coffee_health_10000.csv')
df.head()

,ID,Age,Gender,Country,Coffee_Intake,Caffeine_mg,Sleep_Hours,Sleep_Quality,BMI,Heart_Rate,Stress_Level,Physical_Activity_Hours,Health_Issues,Occupation,Smoking,Alcohol_Consumption
0,1,40,Male,Germany,3.5,328.1,7.5,Good,24.9,78,Low,14.5,NaN,Other,0,0
1,2,33,Male,Germany,1.0,94.1,6.2,Good,20.0,67,Low,11.0,NaN,Service,0,0
2,3,42,Male,Brazil,5.3,503.7,5.9,Fair,22.7,59,Medium,11.2,Mild,Office,0,0
3,4,53,Male,Germany,2.6,249.2,7.3,Good,24.7,71,Low,6.6,Mild,Other,0,0
4,5,32,Female,Spain,3.1,298.0,5.3,Fair,24.1,76,Medium,8.5,Mild,Student,0,1


### Dropeo de Columnas
---

In [3]:
# Quitar columna Health_Issues
df = df.drop(columns=['Health_Issues', 'Caffeine_mg'])
df.head()

,ID,Age,Gender,Country,Coffee_Intake,Sleep_Hours,Sleep_Quality,BMI,Heart_Rate,Stress_Level,Physical_Activity_Hours,Occupation,Smoking,Alcohol_Consumption
0,1,40,Male,Germany,3.5,7.5,Good,24.9,78,Low,14.5,Other,0,0
1,2,33,Male,Germany,1.0,6.2,Good,20.0,67,Low,11.0,Service,0,0
2,3,42,Male,Brazil,5.3,5.9,Fair,22.7,59,Medium,11.2,Office,0,0
3,4,53,Male,Germany,2.6,7.3,Good,24.7,71,Low,6.6,Other,0,0
4,5,32,Female,Spain,3.1,5.3,Fair,24.1,76,Medium,8.5,Student,0,1


Se quito health issues por nulos y se quito caffeine mg por correlacion con coffe intake, ya que eran practicamente lo mismo solo que uno en mg y otro en tazas.

In [4]:
# Filtrar filas donde Gender no sea 'Others'
df = df[df["Gender"] != "Other"]

# Verificar
print(df["Gender"].value_counts())


Gender
Female    5001
Male      4773
Name: count, dtype: int64


### Codificación Variables Categóricas
---

In [5]:
# Map de variables categoricas
df['Sleep_Quality'] = df['Sleep_Quality'].map({'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3})
df['Stress_Level'] = df['Stress_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})
df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})
df = pd.get_dummies(df, columns=['Occupation'], drop_first=True)

In [6]:
df.head()

,ID,Age,Gender,Country,Coffee_Intake,Sleep_Hours,Sleep_Quality,BMI,Heart_Rate,Stress_Level,Physical_Activity_Hours,Smoking,Alcohol_Consumption,Occupation_Office,Occupation_Other,Occupation_Service,Occupation_Student
0,1,40,0,Germany,3.5,7.5,2,24.9,78,0,14.5,0,0,False,True,False,False
1,2,33,0,Germany,1.0,6.2,2,20.0,67,0,11.0,0,0,False,False,True,False
2,3,42,0,Brazil,5.3,5.9,1,22.7,59,1,11.2,0,0,True,False,False,False
3,4,53,0,Germany,2.6,7.3,2,24.7,71,0,6.6,0,0,False,True,False,False
4,5,32,1,Spain,3.1,5.3,1,24.1,76,1,8.5,0,1,False,False,False,True


### Feature Selection
---

In [7]:
# Metodo de envoltura foward feature selection
X = df.drop(columns=['Stress_Level', 'Country', 'ID', 'Occupation'])
y = df['Stress_Level']

def forward_selection_regression(X, y, significance_level=0.05):
    selected_features = []
    remaining_features = list(X.columns)
    
    while remaining_features:
        pvals = pd.Series(index=remaining_features, dtype=float)
        for feature in remaining_features:
            features_to_test = selected_features + [feature]
            X_with_const = sm.add_constant(X[features_to_test])
            model = sm.OLS(y, X_with_const).fit()
            pvals[feature] = model.pvalues[feature]
        min_pval = pvals.min()
        if min_pval < significance_level:
            best_feature = pvals.idxmin()
            selected_features.append(best_feature)
            remaining_features.remove(best_feature)
        else:
            break
    return selected_features

KeyError: "['Occupation'] not found in axis"

In [ ]:
selected_forward = forward_selection_regression(X, y, significance_level=0.05)
print("\n[Envoltura - Forward] Variables seleccionadas:", selected_forward)

ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).

### Balanceo de Variables
---

In [ ]:
df['Stress_Level'].value_counts()